# Watershed Segmentation — Quick Test

Run each cell top to bottom. Last cell shows the result.

In [1]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

IMG_PATH = "../test-img/test5.jpg"  # adjust if needed

img = cv.imread(IMG_PATH)
assert img is not None, f"Cannot read {IMG_PATH}"
img = cv.cvtColor(img, cv.COLOR_BGR2RGB)
gray = cv.cvtColor(img, cv.COLOR_RGB2GRAY)

plt.figure(figsize=(6, 5))
plt.title("Original")
plt.imshow(img)
plt.axis("off")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# 1. Otsu threshold (inverse — pills become white)
ret, thresh = cv.threshold(gray, 0, 255, cv.THRESH_BINARY_INV + cv.THRESH_OTSU)

# 2. Morphological noise cleanup
kernel = np.ones((3, 3), np.uint8)
opening = cv.morphologyEx(thresh, cv.MORPH_OPEN, kernel, iterations=2)

# 3. Sure background (dilate)
sure_bg = cv.dilate(opening, kernel, iterations=3)

# 4. Distance transform → sure foreground
dist = cv.distanceTransform(opening, cv.DIST_L2, 5)
ret, sure_fg = cv.threshold(dist, 0.5 * dist.max(), 255, 0)
sure_fg = np.uint8(sure_fg)

# 5. Unknown region
unknown = cv.subtract(sure_bg, sure_fg)

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
for ax, title, im in zip(
    axes,
    ["Threshold", "Opening", "Distance Transform", "Sure FG markers"],
    [thresh, opening, dist, sure_fg],
):
    ax.set_title(title)
    cmap = "gray" if len(im.shape) == 2 else None
    ax.imshow(im, cmap=cmap)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 6. Label markers
ret, markers = cv.connectedComponents(sure_fg)
markers = markers + 1            # shift so background ≠ 0
markers[unknown == 255] = 0      # mark unknown as 0

# 7. Watershed
markers = cv.watershed(img, markers)

# Count pills (labels ≥ 2, since bg=1 and unknown=0)
pill_count = ret - 1   # connectedComponents returns n_labels including bg

# 8. Draw boundaries on original
result = img.copy()
result[markers == -1] = [255, 0, 0]   # red boundaries

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, title, im, cmap in zip(
    axes,
    ["Original", f"Watershed — {pill_count} pills", "Marker labels"],
    [img, result, markers],
    [None, None, "jet"],
):
    ax.set_title(title, fontsize=13)
    ax.imshow(im, cmap=cmap)
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Detected pills: {pill_count}")

In [ ]:
# Optional: tweak the distance-transform threshold and re-run
# Lower = more markers (may over-segment), Higher = fewer (may merge)
THRESH_FACTOR = 0.5   # try 0.3, 0.4, 0.5, 0.6, 0.7

ret2, sure_fg2 = cv.threshold(dist, THRESH_FACTOR * dist.max(), 255, 0)
sure_fg2 = np.uint8(sure_fg2)
unknown2 = cv.subtract(sure_bg, sure_fg2)
ret2, markers2 = cv.connectedComponents(sure_fg2)
markers2 = markers2 + 1
markers2[unknown2 == 255] = 0
markers2 = cv.watershed(img, markers2)

result2 = img.copy()
result2[markers2 == -1] = [255, 0, 0]

plt.figure(figsize=(7, 6))
plt.title(f"thresh_factor={THRESH_FACTOR} → {ret2 - 1} pills")
plt.imshow(result2)
plt.axis("off")
plt.show()